# EXP-011 / EXP-012 | CatBoost & XGBoost Optuna 튜닝

EXP-006(CatBoost), EXP-007(XGBoost)의 기본 파라미터를 Optuna로 최적화.

| 항목 | 내용 |
|---|---|
| EXP-011 | CatBoost Optuna 튜닝 |
| EXP-012 | XGBoost Optuna 튜닝 |
| 기반 실험 | EXP-006 / EXP-007 |
| CV | Stratified 5-Fold |

In [1]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import warnings
from pathlib import Path
from datetime import date

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, recall_score, precision_score, accuracy_score
from catboost import CatBoostClassifier, Pool
import xgboost as xgb

from src.preprocessing import preprocess

warnings.filterwarnings('ignore')

DATA_DIR = Path('../data/raw')
OUT_DIR  = Path('../data/submissions')
DOCS_DIR = Path('../docs')
TARGET   = '임신 성공 여부'
SEED     = 42
N_FOLDS  = 5
N_TRIALS = 50
AUTHOR   = '조여진'
CV_STR   = f'Stratified {N_FOLDS}-Fold'

train = pd.read_csv(DATA_DIR / 'train.csv')
test  = pd.read_csv(DATA_DIR / 'test.csv')
sub   = pd.read_csv(DATA_DIR / 'sample_submission.csv')
print(f'train: {train.shape}  /  test: {test.shape}')

train: (256351, 69)  /  test: (90067, 68)


## 1. 피처 준비

In [2]:
def add_pre_encode_features(df):
    df = df.copy()
    df['기증_난자_여부'] = (df['난자 출처'] == '기증 제공').astype(int)
    df['기증_정자_여부'] = df['정자 출처'].isin(['기증 제공', '배우자 및 기증 제공']).astype(int)
    return df

def add_derived_features(df):
    df = df.copy()
    eps = 1e-6
    df['수정률']    = df['총 생성 배아 수']   / (df['혼합된 난자 수'] + eps)
    df['이식률']    = df['이식된 배아 수']    / (df['총 생성 배아 수'] + eps)
    df['저장률']    = df['저장된 배아 수']    / (df['총 생성 배아 수'] + eps)
    df['ICSI_비율'] = df['미세주입된 난자 수'] / (df['혼합된 난자 수'] + eps)
    df['배아_발달일']    = df['배아 이식 경과일'] - df['난자 혼합 경과일']
    df['신선_시술_여부']  = df['수집된 신선 난자 수'].notna().astype(int)
    male_cols   = ['남성 주 불임 원인','남성 부 불임 원인','불임 원인 - 남성 요인']
    female_cols = ['여성 주 불임 원인','여성 부 불임 원인','불임 원인 - 난관 질환',
                   '불임 원인 - 배란 장애','불임 원인 - 자궁내막증','불임 원인 - 자궁경부 문제']
    couple_cols = ['부부 주 불임 원인','부부 부 불임 원인']
    sperm_cols  = ['불임 원인 - 정자 농도','불임 원인 - 정자 운동성',
                   '불임 원인 - 정자 형태','불임 원인 - 정자 면역학적 요인']
    all_cause   = male_cols + female_cols + couple_cols + sperm_cols + ['불명확 불임 원인']
    df['남성_불임_합계']      = df[male_cols].sum(axis=1)
    df['여성_불임_합계']      = df[female_cols].sum(axis=1)
    df['부부_불임_합계']      = df[couple_cols].sum(axis=1)
    df['정자_문제_합계']      = df[sperm_cols].sum(axis=1)
    df['총_불임원인_수']      = df[all_cause].sum(axis=1)
    df['임신시도기록_있음']    = df['임신 시도 또는 마지막 임신 경과 연수'].notna().astype(int)
    df['신선_난자_저장_있음']  = (df['저장된 신선 난자 수'] > 0).astype(int)
    df['나이_시술횟수_상호작용'] = df['시술 당시 나이'] * df['총 시술 횟수']
    return df

train_fe = add_pre_encode_features(train)
test_fe  = add_pre_encode_features(test)
X_train, X_test = preprocess(train_fe, test_fe)
X_train = add_derived_features(X_train)
X_test  = add_derived_features(X_test)
y_train = train[TARGET]
print(f'X_train: {X_train.shape}  /  X_test: {X_test.shape}')

X_train: (256351, 85)  /  X_test: (90067, 85)


## 2. CatBoost Optuna 탐색 (EXP-011)

In [3]:
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

def cat_objective(trial):
    params = dict(
        iterations          = 2000,
        learning_rate       = trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        depth               = trial.suggest_int('depth', 4, 10),
        l2_leaf_reg         = trial.suggest_float('l2_leaf_reg', 1.0, 10.0),
        min_data_in_leaf    = trial.suggest_int('min_data_in_leaf', 10, 100),
        subsample           = trial.suggest_float('subsample', 0.6, 1.0),
        colsample_bylevel   = trial.suggest_float('colsample_bylevel', 0.6, 1.0),
        loss_function       = 'Logloss',
        eval_metric         = 'AUC',
        auto_class_weights  = 'Balanced',
        random_seed         = SEED,
        verbose             = False,
        early_stopping_rounds = 50,
    )
    oof = np.zeros(len(X_train))
    for tr_idx, val_idx in skf.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
        model = CatBoostClassifier(**params)
        model.fit(X_tr, y_tr, eval_set=Pool(X_val, y_val), use_best_model=True)
        oof[val_idx] = model.predict_proba(X_val)[:, 1]
    return roc_auc_score(y_train, oof)

cat_study = optuna.create_study(direction='maximize',
                                 sampler=optuna.samplers.TPESampler(seed=SEED))
cat_study.optimize(cat_objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f'\nBest OOF AUC : {cat_study.best_value:.5f}')
print('Best params  :')
for k, v in cat_study.best_params.items():
    print(f'  {k}: {v}')

  0%|          | 0/50 [00:00<?, ?it/s]


Best OOF AUC : 0.74013
Best params  :
  learning_rate: 0.018758723768855998
  depth: 6
  l2_leaf_reg: 9.189608434163782
  min_data_in_leaf: 19
  subsample: 0.8170921295501483
  colsample_bylevel: 0.6936810336930781


## 3. CatBoost 최적 파라미터로 최종 학습

In [4]:
CAT_BEST = dict(
    iterations          = 2000,
    loss_function       = 'Logloss',
    eval_metric         = 'AUC',
    auto_class_weights  = 'Balanced',
    random_seed         = SEED,
    verbose             = False,
    early_stopping_rounds = 100,
    **cat_study.best_params,
)

oof_cat  = np.zeros(len(X_train))
test_cat = np.zeros(len(X_test))
fold_aucs_cat = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
    model = CatBoostClassifier(**CAT_BEST)
    model.fit(X_tr, y_tr, eval_set=Pool(X_val, y_val), use_best_model=True)
    val_prob = model.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, val_prob)
    fold_aucs_cat.append(auc)
    oof_cat[val_idx]  = val_prob
    test_cat         += model.predict_proba(X_test)[:, 1] / N_FOLDS
    print(f'  Fold {fold}  AUC={auc:.5f}')

oof_auc_cat   = roc_auc_score(y_train, oof_cat)
oof_prauc_cat = average_precision_score(y_train, oof_cat)
oof_f1_cat    = f1_score(y_train, (oof_cat >= 0.5).astype(int))
print(f'\nOOF ROC-AUC : {oof_auc_cat:.5f}')
print(f'OOF PR-AUC  : {oof_prauc_cat:.5f}')
print(f'OOF F1      : {oof_f1_cat:.5f}  (threshold=0.5)')
print(f'EXP-006 대비: {oof_auc_cat - 0.74003:+.5f}')

  Fold 1  AUC=0.73818
  Fold 2  AUC=0.74312
  Fold 3  AUC=0.74004
  Fold 4  AUC=0.73866
  Fold 5  AUC=0.74077

OOF ROC-AUC : 0.74015
OOF PR-AUC  : 0.45098
OOF F1      : 0.51740  (threshold=0.5)
EXP-006 대비: +0.00012


## 4. XGBoost Optuna 탐색 (EXP-012)

In [5]:
neg_pos_ratio = (y_train == 0).sum() / (y_train == 1).sum()

def xgb_objective(trial):
    params = dict(
        n_estimators      = 2000,
        learning_rate     = trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
        max_depth         = trial.suggest_int('max_depth', 3, 10),
        min_child_weight  = trial.suggest_int('min_child_weight', 10, 100),
        subsample         = trial.suggest_float('subsample', 0.6, 1.0),
        colsample_bytree  = trial.suggest_float('colsample_bytree', 0.6, 1.0),
        reg_alpha         = trial.suggest_float('reg_alpha', 1e-3, 10.0, log=True),
        reg_lambda        = trial.suggest_float('reg_lambda', 1e-3, 10.0, log=True),
        scale_pos_weight  = neg_pos_ratio,
        eval_metric       = 'auc',
        tree_method       = 'hist',
        random_state      = SEED,
        verbosity         = 0,
        early_stopping_rounds = 50,
    )
    oof = np.zeros(len(X_train))
    for tr_idx, val_idx in skf.split(X_train, y_train):
        X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
        y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
        model = xgb.XGBClassifier(**params)
        model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
        oof[val_idx] = model.predict_proba(X_val)[:, 1]
    return roc_auc_score(y_train, oof)

xgb_study = optuna.create_study(direction='maximize',
                                  sampler=optuna.samplers.TPESampler(seed=SEED))
xgb_study.optimize(xgb_objective, n_trials=N_TRIALS, show_progress_bar=True)

print(f'\nBest OOF AUC : {xgb_study.best_value:.5f}')
print('Best params  :')
for k, v in xgb_study.best_params.items():
    print(f'  {k}: {v}')

  0%|          | 0/50 [00:00<?, ?it/s]


Best OOF AUC : 0.74051
Best params  :
  learning_rate: 0.05520069867907647
  max_depth: 4
  min_child_weight: 59
  subsample: 0.7663066457187595
  colsample_bytree: 0.6581836436885355
  reg_alpha: 8.692038126211928
  reg_lambda: 0.23932562420374562


## 5. XGBoost 최적 파라미터로 최종 학습

In [6]:
XGB_BEST = dict(
    n_estimators     = 2000,
    scale_pos_weight = neg_pos_ratio,
    eval_metric      = 'auc',
    tree_method      = 'hist',
    random_state     = SEED,
    verbosity        = 0,
    early_stopping_rounds = 100,
    **xgb_study.best_params,
)

oof_xgb  = np.zeros(len(X_train))
test_xgb = np.zeros(len(X_test))
fold_aucs_xgb = []

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train), 1):
    X_tr, X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
    y_tr, y_val = y_train.iloc[tr_idx], y_train.iloc[val_idx]
    model = xgb.XGBClassifier(**XGB_BEST)
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=False)
    val_prob = model.predict_proba(X_val)[:, 1]
    auc = roc_auc_score(y_val, val_prob)
    fold_aucs_xgb.append(auc)
    oof_xgb[val_idx]  = val_prob
    test_xgb         += model.predict_proba(X_test)[:, 1] / N_FOLDS
    print(f'  Fold {fold}  AUC={auc:.5f}')

oof_auc_xgb   = roc_auc_score(y_train, oof_xgb)
oof_prauc_xgb = average_precision_score(y_train, oof_xgb)
oof_f1_xgb    = f1_score(y_train, (oof_xgb >= 0.5).astype(int))
print(f'\nOOF ROC-AUC : {oof_auc_xgb:.5f}')
print(f'OOF PR-AUC  : {oof_prauc_xgb:.5f}')
print(f'OOF F1      : {oof_f1_xgb:.5f}  (threshold=0.5)')
print(f'EXP-007 대비: {oof_auc_xgb - 0.74016:+.5f}')

  Fold 1  AUC=0.73854
  Fold 2  AUC=0.74340
  Fold 3  AUC=0.74037
  Fold 4  AUC=0.73872
  Fold 5  AUC=0.74158

OOF ROC-AUC : 0.74051
OOF PR-AUC  : 0.45106
OOF F1      : 0.51790  (threshold=0.5)
EXP-007 대비: +0.00035


## 6. Submission 저장 & 실험 기록

In [7]:
OUT_DIR.mkdir(parents=True, exist_ok=True)

# CatBoost submission
sub_cat = sub.copy()
sub_cat['probability'] = test_cat
auc_str_cat = f'{oof_auc_cat:.4f}'.replace('.', '')
fname_cat = f'submission_exp011_{AUTHOR}_{auc_str_cat}.csv'
sub_cat.to_csv(OUT_DIR / fname_cat, index=False)
print(f'CatBoost 저장: {OUT_DIR / fname_cat}')

# XGBoost submission
sub_xgb = sub.copy()
sub_xgb['probability'] = test_xgb
auc_str_xgb = f'{oof_auc_xgb:.4f}'.replace('.', '')
fname_xgb = f'submission_exp012_{AUTHOR}_{auc_str_xgb}.csv'
sub_xgb.to_csv(OUT_DIR / fname_xgb, index=False)
print(f'XGBoost  저장: {OUT_DIR / fname_xgb}')

CatBoost 저장: ../data/submissions/submission_exp011_조여진_07401.csv
XGBoost  저장: ../data/submissions/submission_exp012_조여진_07405.csv


In [8]:
from openpyxl import load_workbook
from openpyxl.styles import Font, Alignment, Border, Side, PatternFill

def log_to_leaderboard(exp_no, author, model_name, params_str,
                        f1, recall, precision, accuracy, oof_auc,
                        cv_strategy, preprocessing_ver, n_features,
                        imbalance_method, submitted, hackathon_score,
                        file_name, notes='', insights=''):
    lb_path = DOCS_DIR / 'leaderboard.xlsx'
    wb = load_workbook(lb_path)
    ws = wb['리더보드']
    exp_label = f'EXP-{exp_no:03d}'
    next_row = ws.max_row + 1
    for r in range(2, ws.max_row + 1):
        val = ws.cell(row=r, column=2).value
        if val == exp_label:
            next_row = r; break
        if ws.cell(row=r, column=1).value is None or str(ws.cell(row=r, column=1).value).strip() == '':
            next_row = r; break
    values = [str(date.today()), exp_label, author, model_name, params_str,
              round(f1,5), round(recall,5), round(precision,5), round(accuracy,5), round(oof_auc,5),
              cv_strategy, preprocessing_ver, n_features, imbalance_method,
              submitted, hackathon_score, file_name, notes, insights]
    thin = Side(style='thin', color='B0B8D0')
    border = Border(left=thin, right=thin, top=thin, bottom=thin)
    fill = PatternFill('solid', fgColor='EEF2FA') if next_row % 2 == 0 else None
    font = Font(name='맑은 고딕', size=10)
    center = Alignment(horizontal='center', vertical='center', wrap_text=True)
    left   = Alignment(horizontal='left',   vertical='center', wrap_text=True)
    left_cols = {4, 5, 12, 14, 17, 18, 19}
    for c_idx, val in enumerate(values, start=1):
        cell = ws.cell(row=next_row, column=c_idx, value=val)
        cell.font = font; cell.border = border
        cell.alignment = left if c_idx in left_cols else center
        if fill: cell.fill = fill
        if c_idx in range(6, 11) or c_idx == 16: cell.number_format = '0.00000'
    wb.save(lb_path)
    print(f'[leaderboard.xlsx] EXP-{exp_no:03d} 기록 완료 (row {next_row})')

# EXP-011: CatBoost
cat_params_str = ', '.join(
    f'{k}={v:.4g}' if isinstance(v, float) else f'{k}={v}'
    for k, v in cat_study.best_params.items()
)
oof_bin_cat = (oof_cat >= 0.5).astype(int)
log_to_leaderboard(
    11, AUTHOR, 'CatBoost(Optuna)', cat_params_str,
    f1_score(y_train, oof_bin_cat), recall_score(y_train, oof_bin_cat),
    precision_score(y_train, oof_bin_cat), accuracy_score(y_train, oof_bin_cat),
    oof_auc_cat, CV_STR, 'v1', X_train.shape[1], 'auto_class_weights=Balanced',
    'N', None, 'notebooks/10_hparam_cat_xgb_yjcho.ipynb',
    f'CatBoost Optuna {N_TRIALS}trials',
    f'EXP-006 대비 {oof_auc_cat - 0.74003:+.5f}'
)

# EXP-012: XGBoost
xgb_params_str = ', '.join(
    f'{k}={v:.4g}' if isinstance(v, float) else f'{k}={v}'
    for k, v in xgb_study.best_params.items()
)
oof_bin_xgb = (oof_xgb >= 0.5).astype(int)
log_to_leaderboard(
    12, AUTHOR, 'XGBoost(Optuna)', xgb_params_str,
    f1_score(y_train, oof_bin_xgb), recall_score(y_train, oof_bin_xgb),
    precision_score(y_train, oof_bin_xgb), accuracy_score(y_train, oof_bin_xgb),
    oof_auc_xgb, CV_STR, 'v1', X_train.shape[1], f'scale_pos_weight={neg_pos_ratio:.2f}',
    'N', None, 'notebooks/10_hparam_cat_xgb_yjcho.ipynb',
    f'XGBoost Optuna {N_TRIALS}trials',
    f'EXP-007 대비 {oof_auc_xgb - 0.74016:+.5f}'
)

[leaderboard.xlsx] EXP-011 기록 완료 (row 11)
[leaderboard.xlsx] EXP-012 기록 완료 (row 12)


## 7. 튜닝된 파라미터 출력 (앙상블 업데이트용)

아래 파라미터를 `09_ensemble_yjcho.ipynb`의 `CAT_PARAMS` / `XGB_PARAMS`에 붙여넣으세요.

In [9]:
print('=' * 60)
print('▶ CatBoost 최적 파라미터 (09_ensemble에 붙여넣기)')
print('=' * 60)
print('CAT_PARAMS = dict(')
print(f'    iterations=2000, loss_function="Logloss", eval_metric="AUC",')
print(f'    auto_class_weights="Balanced", random_seed={SEED},')
print(f'    verbose=False, early_stopping_rounds=100,')
for k, v in cat_study.best_params.items():
    print(f'    {k}={repr(v)},')
print(')')

print()
print('=' * 60)
print('▶ XGBoost 최적 파라미터 (09_ensemble에 붙여넣기)')
print('=' * 60)
print('XGB_PARAMS = dict(')
print(f'    n_estimators=2000, scale_pos_weight=neg_pos_ratio,')
print(f'    eval_metric="auc", tree_method="hist", random_state={SEED},')
print(f'    verbosity=0, early_stopping_rounds=100,')
for k, v in xgb_study.best_params.items():
    print(f'    {k}={repr(v)},')
print(')')

▶ CatBoost 최적 파라미터 (09_ensemble에 붙여넣기)
CAT_PARAMS = dict(
    iterations=2000, loss_function="Logloss", eval_metric="AUC",
    auto_class_weights="Balanced", random_seed=42,
    verbose=False, early_stopping_rounds=100,
    learning_rate=0.018758723768855998,
    depth=6,
    l2_leaf_reg=9.189608434163782,
    min_data_in_leaf=19,
    subsample=0.8170921295501483,
    colsample_bylevel=0.6936810336930781,
)

▶ XGBoost 최적 파라미터 (09_ensemble에 붙여넣기)
XGB_PARAMS = dict(
    n_estimators=2000, scale_pos_weight=neg_pos_ratio,
    eval_metric="auc", tree_method="hist", random_state=42,
    verbosity=0, early_stopping_rounds=100,
    learning_rate=0.05520069867907647,
    max_depth=4,
    min_child_weight=59,
    subsample=0.7663066457187595,
    colsample_bytree=0.6581836436885355,
    reg_alpha=8.692038126211928,
    reg_lambda=0.23932562420374562,
)
